# PATH MANAGEMENT

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    batch_size=64,
    max_tok_length=16,

    num_shots=5,

    model_name="meta-llama/Meta-Llama-3-8B"
)

# Prompting

Prompting in LLMs is the design of a structured input to provide task description, demostrations and the actual input for the model to generate a desired output.

In this notebook, we are going to use for fine-tuning a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to perform In-Context Learning with the [Llama2 model](https://huggingface.co/docs/transformers/model_doc/llama2) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

In [3]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 200965
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [4]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [5]:
raw_datasets["train"][:14]["source_text"]

['artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 'en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.',
 'hubo unos 200 invitados.',
 '¿eres tú mayor de edad?',
 '-"pero no tienes dinero, ¿verdad?"',
 'así que, cuando ella te deja, ¿de donde crees que ella va a hacer a continuación.',
 'para empezar, como ya hemos dicho, debemos tomar la fruta con el estómago vacío.',
 'soy una viuda con cuatro hijos y me quedé atrapado en una situación financiera desde abril de 2016 y necesitaba refinanciar y pagar mis cuentas.',
 'el trabajo con espacios en blanco debe comenzar a fines de la primavera o principios del verano y no retrasarse hasta el otoño para evitar problemas e interrupciones.',
 'es la riqueza guardada por su dueño para su propia desgracia.',
 'buscamos una canción que trate sobre alguno de los siguientes temas: «desarrollo global» o «un solo

In [6]:
raw_datasets["train"][:14]["dest_text"]

['estis mistero por la polico : kial ŝteli nur unu ŝuon anstataù paro ?',
 'la tria jarcento vidis la aperon de kelkaj grandaj okcident ĝermanaj triboj: la alemanoj, frankoj, bavarii-, ĥatoj, saksoj, frisii, sicambri, kaj thuringii.',
 'venis ĉirkaŭ 200 gastoj.',
 'ĉu vi estas la plej aĝa?',
 '"sed vi ne posedas tiom da mono, ĉu ne?"',
 'do, kiam ŝi lasas vin, kie vi kredas, ke ŝi faros poste.',
 'kiel antaŭe menciite, la drogo devas esti prenita sur malplena stomako.',
 'en ĉi tiu tempo mi estas vidvino kun kvar infanoj kaj mi estis ligita en financa situacio en majo 2018 kaj bezonis refinanci kaj pagi miajn biletojn.',
 'laboro kun spacoj devas komenciĝi fine de printempo aŭ frua somero kaj ne malhelpu ĝis aŭtuno por eviti problemojn kaj interrompojn.',
 'riĉecon konservatan por la malutilo de ĝia propra mastro.',
 'tie ĉi mi menciu nur unu temaron, tiun de tutmondiĝo aŭ „globaliĝo”.',
 'ni serĉu rekte la titolon «orientaj tapiŝoj»!',
 'tamen, ĉu tranĉeoj estas por ke ni koncentriĝu'

In [7]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

The Llama2 model is a pretrained Large Language Model (LLM) ready to tackle several NLP tasks, being one of the them the translation from English into Spanish. Let us filter the Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [8]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

More precisely, we are going to be using the Llama-2 checkpoint [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) to run our experiments for which you need to accept the LLAMA 2 COMMUNITY LICENSE AGREEMENT. Processing your request may take some time, so please do it in advance.

Logging in HuggingFace to be granted access to Llama2 with 7B parameters:

In [9]:
import os
import dotenv
from huggingface_hub import login

dotenv.load_dotenv()
login(token=os.getenv("HUGGING_FACE_TOKEN"))

python-dotenv could not parse statement starting at line 2


python-dotenv could not parse statement starting at line 4


python-dotenv could not parse statement starting at line 5


python-dotenv could not parse statement starting at line 7


python-dotenv could not parse statement starting at line 8


We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the needs of the model that is to be prompted. In the case of Llama2, it is recommended to explicitly state a task prompt for each source sentence:

In [10]:
from transformers import AutoTokenizer

checkpoint = CONFIG.model_name #"meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    token=True,
    padding=True,
    pad_to_multiple_of=8,
    truncation=True,
    max_length=CONFIG.max_tok_length,
    padding_side='left',
    )
tokenizer.pad_token = tokenizer.eos_token

In [11]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [12]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[128000, 472, 49591, 37229, 325, 90870, 64, 653, 296, 1601, 822, 25, 29386, 4406, 43388, 1647, 72, 1417, 17996, 11639, 409, 311, 267, 2172, 30], [128000, 268, 658, 8531, 385, 63193, 12274, 40608, 653, 31311, 409, 14121, 355, 82986, 11644, 15540, 1624, 297, 18223, 38546, 25, 264, 3516, 43761, 11, 44579, 437, 11, 8415, 437, 11, 76880, 82, 1662, 3233, 11, 1448, 285, 3893, 11, 76880, 52877, 3042, 462, 11, 379, 270, 1711, 3893, 3305, 13]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[128000, 478, 285, 296, 1601, 78, 4247, 1208, 1499, 4042, 551, 597, 532, 27006, 251, 83, 12574, 12500, 653, 84, 27006, 251, 84, 263, 459, 267, 460, 15273, 1370, 78, 949], [128000, 4355, 2463, 64, 30695, 1189, 78, 18619, 285, 1208, 264, 716, 263, 409, 49328, 74, 1662, 6800, 1662, 550

In [13]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['<|begin_of_text|>', 'art', 'ÃŃculo', 'Ġanterior', 'se', 'Ġdevel', 'a', 'Ġun', 'Ġm', 'ister', 'io', ':', 'ĠÂ¿', 'por', 'ĠquÃ©', 'Ġmon', 'i', 'Ġarg', 'ento', 'Ġera', 'Ġde', 'Ġto', 'st', 'ado', '?']
['<|begin_of_text|>', 'en', 'Ġel', 'Ġsig', 'lo', 'Ġiii', 'Ġsurg', 'ieron', 'Ġun', 'ĠnÃºmero', 'Ġde', 'Ġtrib', 'us', 'Ġgerm', 'Ã¡n', 'icas', 'Ġdel', 'Ġo', 'este', 'Ġgrandes', ':', 'Ġa', 'lem', 'anni', ',', 'Ġfranc', 'os', ',', 'Ġcat', 'os', ',', 'Ġï»¿', 's', 'aj', 'ones', ',', 'Ġfr', 'is', 'ii', ',', 'Ġï»¿', 'sic', 'amb', 'ri', ',', 'Ġy', 'Ġth', 'uring', 'ii', 'ï»¿', '.']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [14]:
tokenizer.batch_decode(model_input['input_ids'])

['<|begin_of_text|>artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 '<|begin_of_text|>en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [15]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|                                                        | 0/200965 [00:00<?, ? examples/s]

Map:   3%|█▎                                        | 6000/200965 [00:00<00:06, 28215.24 examples/s]

Map:   7%|██▊                                      | 14000/200965 [00:00<00:04, 43693.87 examples/s]

Map:  11%|████▍                                    | 22000/200965 [00:00<00:04, 37857.83 examples/s]

Map:  15%|██████                                   | 30000/200965 [00:00<00:03, 45216.21 examples/s]

...

Map:  39%|████████████████▌                         | 17000/43063 [00:00<00:00, 42180.88 examples/s]

Map:  58%|████████████████████████▍                 | 25000/43063 [00:00<00:00, 50775.15 examples/s]

Map:  72%|██████████████████████████████▏           | 31000/43063 [00:00<00:00, 40175.59 examples/s]

Map:  91%|██████████████████████████████████████    | 39000/43063 [00:00<00:00, 47252.55 examples/s]

Map: 100%|██████████████████████████████████████████| 43063/43063 [00:01<00:00, 41998.52 examples/s]

We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [16]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 16 tokens:   0%| | 0/200965 [00:00<?, ? exampl

Discarding source and target sentences with more than 16 tokens:   4%| | 8000/200965 [00:00<00:02, 6

Discarding source and target sentences with more than 16 tokens:   8%| | 16000/200965 [00:00<00:02, 

Discarding source and target sentences with more than 16 tokens:  12%| | 24000/200965 [00:00<00:02, 

Discarding source and target sentences with more than 16 tokens:  16%|▏| 32000/200965 [00:00<00:02, 

...

Discarding source and target sentences with more than 16 tokens:  37%|▎| 16000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens:  56%|▌| 24000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens:  74%|▋| 32000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens:  93%|▉| 40000/43063 [00:00<00:00, 6

Discarding source and target sentences with more than 16 tokens: 100%|█| 43063/43063 [00:00<00:00, 6

We can take a quick look at the length histogram in the source language:

In [17]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 3  13
 4 293
 5 1423
 6 3914
 7 7463
 8 10260
 9 11170
10 10333
11 8766
12 6586
13 4502
14 2996
15 1832
16 1152


Checking a sample after filtering by maximum number of tokens:

In [18]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[128000, 27780, 78, 53998, 220, 1049, 83167, 5670, 13]
[1, 1, 1, 1, 1, 1, 1, 1, 1]
[128000, 1055, 285, 10044, 231, 404, 4657, 129, 255, 220, 1049, 88759, 21963, 13]
[128000, 31282, 13213, 90318, 17352, 409, 53782, 30]
[1, 1, 1, 1, 1, 1, 1, 1]
[128000, 128, 231, 84, 3355, 48591, 1208, 7245, 73, 264, 128, 251, 64, 30]
[128000, 6455, 912, 19972, 288, 781, 1744, 15833, 6502, 4988, 11, 513, 64367, 409, 66908, 13]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[128000, 325, 389, 72, 841, 1156, 3557, 12776, 597, 822, 85006, 11, 85006, 12776, 40774, 13]
[128000, 22025, 24012, 33482, 2172, 1174, 26976, 33558]
[1, 1, 1, 1, 1, 1, 1, 1]
[128000, 3315, 129, 255, 969, 21488, 11, 5899, 2172]
[128000, 265, 3394, 409, 3920, 511, 689, 320, 71, 14635, 220, 8258, 22, 8]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[128000, 526, 385, 17190, 822, 320, 128, 251, 285, 220, 8258, 22, 8]


In [19]:
src = CONFIG.src_abr
tgt = CONFIG.tgt_abr
task_prefix = f"Translate from {src} to {tgt}:\n"
shots = ""
s = ""

prefix_tok_len = len(tokenizer.encode(f"{task_prefix}{shots}{src}: {s} = {tgt}: "))
shot_tok_len   = len(tokenizer.encode(f"{src}: {s} = {tgt}: {s}\n"))
max_tok_len = prefix_tok_len
max_tok_len += CONFIG.num_shots * (shot_tok_len + 2 * CONFIG.max_tok_length) 
max_tok_len += CONFIG.max_tok_length

random_seed = 13
sample = tokenized_datasets['train'].shuffle(seed=random_seed).select(range(CONFIG.num_shots))
for s in sample: shots += f"{src}: {s['source_text']} = {tgt}: {s['dest_text']}\n" 

def preprocess4test_function(sample):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_tok_len, 
        truncation=True, 
        # return_tensors="pt", 
        # padding=True)
        padding=False,  # Changed to False - don't pad during preprocessing
        return_tensors=None)  # Changed to None - return lists instead of tensors
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*:

In [20]:
sample = tokenized_datasets['test'].select(range(5))
model_input = preprocess4test_function(sample)
print(model_input)
print(tokenizer.batch_decode(model_input['input_ids']))

{'input_ids': [[128000, 28573, 505, 1560, 311, 95703, 512, 288, 25, 938, 5683, 906, 5888, 25, 1560, 653, 824, 299, 25626, 13, 284, 95703, 25, 12751, 17588, 6632, 73, 25, 1689, 64, 305, 13807, 627, 288, 25, 4689, 12826, 1560, 7495, 2649, 665, 2362, 3008, 552, 1974, 13, 284, 95703, 25, 259, 822, 48591, 31699, 547, 5899, 6388, 665, 2362, 3008, 552, 1974, 627, 288, 25, 49913, 385, 30234, 15295, 3429, 68681, 2092, 437, 0, 284, 95703, 25, 10044, 251, 285, 342, 1662, 37491, 10006, 4247, 9686, 96189, 4999, 288, 25, 12769, 31282, 64591, 4698, 276, 5252, 97673, 30, 284, 95703, 25, 1054, 74, 822, 1879, 333, 300, 1208, 82837, 566, 2058, 21963, 12671, 198, 288, 25, 20651, 409, 23200, 18745, 25, 16, 12, 23, 284, 95703, 25, 308, 2925, 299, 409, 45064, 13873, 73, 25, 220, 23, 12, 972, 198, 288, 25, 671, 299, 2537, 297, 40261, 11, 951, 108606, 1347, 2172, 13, 284, 95703, 25, 220], [128000, 28573, 505, 1560, 311, 95703, 512, 288, 25, 938, 5683, 906, 5888, 25, 1560, 653, 824, 299, 25626, 13, 284, 95703, 

In [21]:
preprocessed_test_dataset = tokenized_datasets['test'].map(preprocess4test_function, batched=True)

Map:   0%|                                                         | 0/15358 [00:00<?, ? examples/s]

Map:  13%|█████▋                                      | 2000/15358 [00:00<00:01, 9738.41 examples/s]

Map:  39%|████████████████▊                          | 6000/15358 [00:00<00:00, 17252.40 examples/s]

Map:  65%|███████████████████████████▎              | 10000/15358 [00:00<00:00, 20083.64 examples/s]

Map:  91%|██████████████████████████████████████▎   | 14000/15358 [00:00<00:00, 21523.45 examples/s]

Map: 100%|██████████████████████████████████████████| 15358/15358 [00:00<00:00, 19850.55 examples/s]

In [22]:
for sample in preprocessed_test_dataset.select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[128000, 28573, 505, 1560, 311, 95703, 512, 288, 25, 938, 5683, 906, 5888, 25, 1560, 653, 824, 299, 25626, 13, 284, 95703, 25, 12751, 17588, 6632, 73, 25, 1689, 64, 305, 13807, 627, 288, 25, 4689, 12826, 1560, 7495, 2649, 665, 2362, 3008, 552, 1974, 13, 284, 95703, 25, 259, 822, 48591, 31699, 547, 5899, 6388, 665, 2362, 3008, 552, 1974, 627, 288, 25, 49913, 385, 30234, 15295, 3429, 68681, 2092, 437, 0, 284, 95703, 25, 10044, 251, 285, 342, 1662, 37491, 10006, 4247, 9686, 96189, 4999, 288, 25, 12769, 31282, 64591, 4698, 276, 5252, 97673, 30, 284, 95703, 25, 1054, 74, 822, 1879, 333, 300, 1208, 82837, 566, 2058, 21963, 12671, 198, 288, 25, 20651, 409, 23200, 18745, 25, 16, 12, 23, 284, 95703, 25, 308, 2925, 299, 409, 45064, 13873, 73, 25, 220, 23, 12, 972, 198, 288, 25, 671, 299, 2537, 297, 40261, 11, 951, 108606, 1347, 2172, 13, 284, 95703, 25, 220]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [23]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [24]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    token=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
)


Loading checkpoint shards:   0%|                                              | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|█████████▌                            | 1/4 [00:01<00:04,  1.55s/it]

Loading checkpoint shards:  50%|███████████████████                   | 2/4 [00:03<00:03,  1.73s/it]

Loading checkpoint shards:  75%|████████████████████████████▌         | 3/4 [00:05<00:01,  1.76s/it]

Loading checkpoint shards: 100%|██████████████████████████████████████| 4/4 [00:05<00:00,  1.12s/it]

Loading checkpoint shards: 100%|██████████████████████████████████████| 4/4 [00:05<00:00,  1.33s/it]

# Inference

Loading default inference parameters for the model, so that additional parameters could be added and passed to the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation):

In [25]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
    )

print(generation_config)

GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "max_length": 4096,
  "temperature": 0.6,
  "top_p": 0.9
}



As observed, the default search strategy for Llama-2 is Top-p with probability 0.9 and temperature 0.6 ($0<T<1$ amplifies output probability differences and makes output more deterministic). [The search strategy can be selected](https://huggingface.co/docs/transformers/en/generation_strategies) at inference time. 

First, the test set is divided in small batches to reduce GPU memory comsumption:

In [26]:
batch_tokenized_test = preprocessed_test_dataset.batch(CONFIG.batch_size)

Batching examples:   0%|                                           | 0/15358 [00:00<?, ? examples/s]

Batching examples:  11%|███▏                         | 1664/15358 [00:00<00:00, 15774.76 examples/s]

Batching examples:  22%|██████▎                      | 3328/15358 [00:00<00:00, 15755.40 examples/s]

Batching examples:  33%|█████████▍                   | 4992/15358 [00:00<00:00, 15765.83 examples/s]

Batching examples:  43%|████████████▍                | 6592/15358 [00:00<00:00, 15598.63 examples/s]

Batching examples:  54%|███████████████▌             | 8256/15358 [00:00<00:00, 15668.26 examples/s]

Batching examples:  65%|██████████████████▋          | 9920/15358 [00:00<00:00, 15742.61 examples/s]

Batching examples:  75%|█████████████████████       | 11584/15358 [00:00<00:00, 15766.28 examples/s]

Batching examples:  91%|█████████████████████████▍  | 13952/15358 [00:00<00:00, 15482.98 examples/s]

Batching examples: 100%|████████████████████████████| 15358/15358 [00:00<00:00, 15609.84 examples/s]

Por alguna razón en algunos batches se quedaban secuencias con diferentes tamaños. He tenido que darles padding tras generarlos.

In [27]:
import tqdm 

# Get newline token ID for early stopping
newline_token_id = tokenizer.encode("\n", add_special_tokens=False)[0]

number_of_batches = len(batch_tokenized_test["input_ids"])
output_sequences = []
all_sources=[]

for i in tqdm.tqdm(range(number_of_batches)):
    all_sources.append(batch_tokenized_test["input_ids"][i])
    
    # Pad the batch to the same length to avoid inconsistent lengths in a batch
    batch_inputs = tokenizer.pad(
        {"input_ids": batch_tokenized_test["input_ids"][i], 
         "attention_mask": batch_tokenized_test["attention_mask"][i]},
        padding=True,
        return_tensors="pt"
    )
    
    with torch.no_grad():
        output_batch = model.generate(
            input_ids=batch_inputs["input_ids"].cuda(), 
            attention_mask=batch_inputs["attention_mask"].cuda(), 
            max_new_tokens=CONFIG.max_tok_length * 2,  # Allow reasonable generation length
            min_new_tokens=1,  # Ensure at least some output
            num_beams=1, 
            do_sample=False,
            eos_token_id=[tokenizer.eos_token_id, newline_token_id],  # Stop at EOS or newline
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.2,  # Discourage repetition
        )
    output_sequences.extend(output_batch)


  0%|                                                                       | 0/240 [00:00<?, ?it/s]

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  0%|▎                                                              | 1/240 [00:06<26:26,  6.64s/it]

  1%|▌                                                              | 2/240 [00:13<25:56,  6.54s/it]

...

 99%|████████████████████████████████████████████████████████████▏| 237/240 [25:34<00:19,  6.47s/it]

 99%|████████████████████████████████████████████████████████████▍| 238/240 [25:41<00:12,  6.48s/it]

100%|████████████████████████████████████████████████████████████▋| 239/240 [25:47<00:06,  6.48s/it]

100%|█████████████████████████████████████████████████████████████| 240/240 [25:54<00:00,  6.44s/it]

100%|█████████████████████████████████████████████████████████████| 240/240 [25:54<00:00,  6.48s/it]

## Evaluation

The output of the model is automatically evaluated compared to the reference translations. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [28]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|                                                       | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|███████████████████████████████████████████| 5/5 [00:00<00:00, 106997.55it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


The example below performs a basic post-processing to decode the predictions and extract the translation:

In [29]:
import re

def compute_metrics(sample, output_sequences):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    # print(inputs)
    # print(preds)
    for i, (input,pred) in enumerate(zip(inputs,preds)):
      pred = re.search(r'^.*\n',pred.removeprefix(input).lstrip())
      if pred is not None:
        preds[i] = pred.group()[:-1]
      else:
        preds[i] = ""
    # print(sample["source_text"])
    # print(sample["dest_text"])
    # print(preds)
    result_bleu = metric_bleu.compute(
       predictions=preds, 
       references=sample["dest_text"]
    )
    result_comet = metric_comet.compute(
        sources=sample["source_text"],
        predictions=preds, 
        references=sample["dest_text"]
    )
    result = {
      "bleu": result_bleu["score"],
      "comet": result_comet["mean_score"]
      }
    return result

In [30]:
result = compute_metrics(preprocessed_test_dataset,output_sequences, )
print(f'BLEU score: {result["bleu"]:0.4f}')
print(f'COMET score: {result["comet"]:0.4f}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/t

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


BLEU score: 7.9754
COMET score: 0.6624


In [31]:
from maikol_utils.print_utils import print_separator

# all_sources already contains raw text strings
# output_sequences contains token IDs that need to be decoded
decoded_outputs = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

# Get reference translations from test set
test_references = tokenized_datasets["test"]["dest_text"]

# Print first 10 examples
for i, (source, output, reference) in enumerate(zip(all_sources[:10], decoded_outputs[:10], test_references[:10])):
    print_separator(f"Example {i+1}:")
    print(f"Source:      {source}")
    print(f"Translation: {output}")
    print(f"Reference:   {reference}")

________________________________________________________________
                           Example 1:                           

Source:      [[128000, 28573, 505, 1560, 311, 95703, 512, 288, 25, 938, 5683, 906, 5888, 25, 1560, 653, 824, 299, 25626, 13, 284, 95703, 25, 12751, 17588, 6632, 73, 25, 1689, 64, 305, 13807, 627, 288, 25, 4689, 12826, 1560, 7495, 2649, 665, 2362, 3008, 552, 1974, 13, 284, 95703, 25, 259, 822, 48591, 31699, 547, 5899, 6388, 665, 2362, 3008, 552, 1974, 627, 288, 25, 49913, 385, 30234, 15295, 3429, 68681, 2092, 437, 0, 284, 95703, 25, 10044, 251, 285, 342, 1662, 37491, 10006, 4247, 9686, 96189, 4999, 288, 25, 12769, 31282, 64591, 4698, 276, 5252, 97673, 30, 284, 95703, 25, 1054, 74, 822, 1879, 333, 300, 1208, 82837, 566, 2058, 21963, 12671, 198, 288, 25, 20651, 409, 23200, 18745, 25, 16, 12, 23, 284, 95703, 25, 308, 2925, 299, 409, 45064, 13873, 73, 25, 220, 23, 12, 972, 198, 288, 25, 671, 299, 2537, 297, 40261, 11, 951, 108606, 1347, 2172, 13, 284, 95703, 25,

<span style="color:lightgreen">
Como se le dan ejemplos, la salida (translation) tiene estos ejemplos + la predicción en el último
</span>